# Phase 5: Visualization & EDA

**Goal:** Understand the data visually before modelling. Every chart answers a specific question.

**9 charts:**
1. Team Win Rates (all-time)
2. Toss Impact by Venue
3. Head-to-Head Heatmap
4. Recent Form vs Actual Wins
5. Feature Correlation Heatmap
6. Venue Analysis (bat first vs chase)
7. Player Impact (batting SR: winners vs losers)
8. Season Progression
9. Player Matchup Heatmap (MI vs CSK)

All charts saved as PNGs in `notebooks/charts/`

## 1. Imports & Load

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings, os
warnings.filterwarnings('ignore')

os.makedirs('charts', exist_ok=True)

# Use a clean dark style — looks like broadcast graphics
plt.style.use('dark_background')
ACCENT   = '#00D4FF'   # cyan
ACCENT2  = '#FF6B35'   # orange
GREEN    = '#2ECC71'
RED      = '#E74C3C'

full     = pd.read_csv('../data/processed/full_features.csv', parse_dates=['date'])
matches  = pd.read_csv('../data/processed/cleaned_matches.csv', parse_dates=['date'])
matchups = pd.read_csv('../data/processed/player_matchups.csv')

# Active teams only (exclude defunct franchises for cleaner charts)
ACTIVE_TEAMS = [
    'Mumbai Indians', 'Chennai Super Kings', 'Royal Challengers Bengaluru',
    'Kolkata Knight Riders', 'Delhi Capitals', 'Rajasthan Royals',
    'Punjab Kings', 'Sunrisers Hyderabad', 'Gujarat Titans', 'Lucknow Super Giants'
]

print(f'Loaded: {len(full)} matches, {len(matchups):,} matchup pairs')

## Chart 1: Team Win Rates (All-Time)
**Question:** Which teams are historically the strongest?

In [ ]:
# Count wins and total matches for each team
team_stats = {}
for team in ACTIVE_TEAMS:
    team_matches = full[(full['team_a'] == team) | (full['team_b'] == team)]
    wins = (team_matches['team_a_won'].values * (team_matches['team_a'] == team).values +
            (1 - team_matches['team_a_won'].values) * (team_matches['team_b'] == team).values).sum()
    team_stats[team] = {
        'matches': len(team_matches),
        'wins'   : int(wins),
        'win_rate': wins / len(team_matches) if len(team_matches) > 0 else 0
    }

stats_df = pd.DataFrame(team_stats).T.sort_values('win_rate', ascending=True)

fig, ax = plt.subplots(figsize=(10, 7))
bars = ax.barh(stats_df.index, stats_df['win_rate'], color=ACCENT, alpha=0.85, height=0.6)

# Add win rate labels
for bar, (_, row) in zip(bars, stats_df.iterrows()):
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
            f"{row['win_rate']:.1%}  ({int(row['wins'])}/{int(row['matches'])})",
            va='center', fontsize=9, color='white')

ax.axvline(0.5, color='white', linestyle='--', alpha=0.4, linewidth=1)
ax.set_xlim(0, 0.75)
ax.set_xlabel('Win Rate', fontsize=12)
ax.set_title('IPL All-Time Win Rates (Active Teams)', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('charts/01_team_win_rates.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: charts/01_team_win_rates.png')

## Chart 2: Toss Impact by Venue
**Question:** Does winning the toss matter more at certain venues?

In [ ]:
# For each venue, compute: bat-first win rate and toss-winner win rate
top_venues = matches['venue'].value_counts().head(10).index.tolist()

venue_toss = []
for venue in top_venues:
    vm = full[full['venue'] == venue]
    if len(vm) < 10:
        continue
    bat_first_wr = vm['team_a_won'].mean()   # team_a always bats first
    toss_won_match = (vm['toss_winner_is_ta'] == vm['team_a_won']) | \
                     ((vm['toss_winner_is_ta'] == 0) & (vm['team_a_won'] == 0))
    toss_wr = toss_won_match.mean()
    venue_toss.append({'venue': venue.split(' ')[0], 'bat_first_wr': bat_first_wr, 'toss_win_wr': toss_wr, 'matches': len(vm)})

vt_df = pd.DataFrame(venue_toss).sort_values('bat_first_wr')

x = np.arange(len(vt_df))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 6))
ax.bar(x - width/2, vt_df['bat_first_wr'], width, label='Bat First Win Rate', color=ACCENT, alpha=0.85)
ax.bar(x + width/2, vt_df['toss_win_wr'],  width, label='Toss Winner Win Rate', color=ACCENT2, alpha=0.85)
ax.axhline(0.5, color='white', linestyle='--', alpha=0.4)
ax.set_xticks(x)
ax.set_xticklabels(vt_df['venue'], rotation=30, ha='right', fontsize=9)
ax.set_ylabel('Win Rate')
ax.set_title('Bat-First vs Toss Winner Win Rate by Venue', fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('charts/02_toss_impact_venue.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: charts/02_toss_impact_venue.png')

## Chart 3: Head-to-Head Heatmap
**Question:** Which matchups are most one-sided historically?

In [ ]:
SHORT = {
    'Mumbai Indians': 'MI', 'Chennai Super Kings': 'CSK',
    'Royal Challengers Bengaluru': 'RCB', 'Kolkata Knight Riders': 'KKR',
    'Delhi Capitals': 'DC', 'Rajasthan Royals': 'RR',
    'Punjab Kings': 'PBKS', 'Sunrisers Hyderabad': 'SRH',
    'Gujarat Titans': 'GT', 'Lucknow Super Giants': 'LSG'
}

teams = list(SHORT.keys())
h2h_matrix = pd.DataFrame(np.nan, index=[SHORT[t] for t in teams], columns=[SHORT[t] for t in teams])

for ta in teams:
    for tb in teams:
        if ta == tb:
            continue
        h2h_matches = full[
            ((full['team_a'] == ta) & (full['team_b'] == tb)) |
            ((full['team_a'] == tb) & (full['team_b'] == ta))
        ]
        if len(h2h_matches) > 0:
            wins_ta = (
                (h2h_matches['team_a'] == ta) & (h2h_matches['team_a_won'] == 1) |
                (h2h_matches['team_b'] == ta) & (h2h_matches['team_a_won'] == 0)
            ).sum()
            h2h_matrix.loc[SHORT[ta], SHORT[tb]] = round(wins_ta / len(h2h_matches), 2)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(h2h_matrix.astype(float), annot=True, fmt='.2f', cmap='RdYlGn',
            vmin=0, vmax=1, center=0.5, ax=ax,
            linewidths=0.5, cbar_kws={'label': 'Win Rate (row team)'})
ax.set_title('Head-to-Head Win Rates (row team vs column team)', fontsize=13, fontweight='bold', pad=15)
ax.set_xlabel('Opposition')
ax.set_ylabel('Team')
plt.tight_layout()
plt.savefig('charts/03_h2h_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: charts/03_h2h_heatmap.png')

## Chart 4: Recent Form vs Actual Wins
**Question:** Does last-5-match form actually predict who wins?

In [ ]:
# Compute form difference (team_a form minus team_b form)
full['form_diff'] = full['ta_last5_wr'] - full['tb_last5_wr']

# Bin into form buckets
bins   = [-1, -0.4, -0.2, 0.0, 0.2, 0.4, 1]
labels = ['<<B', '<B', '≈B', '≈A', '>A', '>>A']
full['form_bucket'] = pd.cut(full['form_diff'], bins=bins, labels=labels)

bucket_wr = full.groupby('form_bucket', observed=True)['team_a_won'].agg(['mean','count']).reset_index()

fig, ax = plt.subplots(figsize=(9, 5))
colors = [RED, RED, 'gray', 'gray', GREEN, GREEN]
bars = ax.bar(bucket_wr['form_bucket'].astype(str), bucket_wr['mean'], color=colors, alpha=0.85)
for bar, (_, row) in zip(bars, bucket_wr.iterrows()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f"{row['mean']:.1%}\n(n={int(row['count'])})",
            ha='center', fontsize=9)
ax.axhline(0.5, color='white', linestyle='--', alpha=0.4)
ax.set_ylim(0, 0.8)
ax.set_xlabel('Form Advantage  (A = team_a, B = team_b)')
ax.set_ylabel('Team A Win Rate')
ax.set_title('Does Recent Form Predict Match Outcome?', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('charts/04_form_vs_wins.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: charts/04_form_vs_wins.png')

## Chart 5: Feature Correlation Heatmap
**Question:** Which features are most correlated with winning? Which are redundant?

In [ ]:
feature_cols = [
    'ta_overall_wr','tb_overall_wr','ta_venue_wr','tb_venue_wr','ta_h2h_wr',
    'ta_last5_wr','tb_last5_wr','ta_streak','tb_streak',
    'toss_winner_is_ta','venue_batfirst_wr','ta_home',
    'ta_season_wr','tb_season_wr','is_playoff',
    'ta_avg_batting_sr','tb_avg_batting_sr',
    'ta_avg_bowling_econ','tb_avg_bowling_econ',
    'ta_total_caps','tb_total_caps',
    'ta_favorable_matchups','tb_favorable_matchups',
    'venue_avg_first_inn_score',
    'team_a_won'
]

corr = full[feature_cols].corr()

fig, ax = plt.subplots(figsize=(14, 12))
mask = np.triu(np.ones_like(corr, dtype=bool))   # hide upper triangle (duplicate)
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            vmin=-1, vmax=1, center=0, ax=ax, linewidths=0.3,
            annot_kws={'size': 7})
ax.set_title('Feature Correlation Matrix', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('charts/05_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: charts/05_correlation_heatmap.png')

# Print top features correlated with target
print('\nTop features correlated with team_a_won:')
print(corr['team_a_won'].drop('team_a_won').abs().sort_values(ascending=False).head(10))

## Chart 6: Venue Analysis
**Question:** Which venues heavily favour batting first vs chasing?

In [ ]:
venue_analysis = []
for venue in top_venues:
    vm = full[full['venue'] == venue]
    if len(vm) < 8: continue
    venue_analysis.append({
        'venue'         : venue.replace(' Stadium','').replace(' Cricket Association',''),
        'bat_first_wr'  : vm['team_a_won'].mean(),
        'chase_wr'      : 1 - vm['team_a_won'].mean(),
        'avg_score'     : vm['venue_avg_first_inn_score'].mean(),
        'matches'       : len(vm)
    })

va_df = pd.DataFrame(venue_analysis).sort_values('bat_first_wr', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: bat first win rate
colors = [GREEN if x > 0.5 else RED for x in va_df['bat_first_wr']]
axes[0].barh(va_df['venue'], va_df['bat_first_wr'], color=colors, alpha=0.85)
axes[0].axvline(0.5, color='white', linestyle='--', alpha=0.5)
axes[0].set_xlabel('Bat-First Win Rate')
axes[0].set_title('Bat-First Win Rate by Venue', fontweight='bold')

# Right: average first innings score
axes[1].barh(va_df['venue'], va_df['avg_score'], color=ACCENT, alpha=0.85)
axes[1].set_xlabel('Avg First Innings Score')
axes[1].set_title('Avg First Innings Score by Venue', fontweight='bold')

plt.suptitle('Venue Analysis', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('charts/06_venue_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: charts/06_venue_analysis.png')

## Chart 7: Player Impact — Batting SR in Winning vs Losing Teams
**Question:** Do winning teams have higher batting strike rates?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Batting SR distribution: winners vs losers
winners = full[full['team_a_won'] == 1]['ta_avg_batting_sr']
losers  = full[full['team_a_won'] == 0]['ta_avg_batting_sr']

axes[0].hist(winners, bins=25, alpha=0.7, color=GREEN,  label=f'Winners  (mean={winners.mean():.1f})')
axes[0].hist(losers,  bins=25, alpha=0.7, color=RED,    label=f'Losers   (mean={losers.mean():.1f})')
axes[0].axvline(winners.mean(), color=GREEN, linestyle='--')
axes[0].axvline(losers.mean(),  color=RED,   linestyle='--')
axes[0].set_xlabel('Team Avg Batting Strike Rate')
axes[0].set_ylabel('Number of Matches')
axes[0].set_title('Batting SR: Winners vs Losers', fontweight='bold')
axes[0].legend()

# Bowling economy distribution
winners_econ = full[full['team_a_won'] == 1]['ta_avg_bowling_econ']
losers_econ  = full[full['team_a_won'] == 0]['ta_avg_bowling_econ']

axes[1].hist(winners_econ, bins=25, alpha=0.7, color=GREEN, label=f'Winners  (mean={winners_econ.mean():.1f})')
axes[1].hist(losers_econ,  bins=25, alpha=0.7, color=RED,   label=f'Losers   (mean={losers_econ.mean():.1f})')
axes[1].axvline(winners_econ.mean(), color=GREEN, linestyle='--')
axes[1].axvline(losers_econ.mean(),  color=RED,   linestyle='--')
axes[1].set_xlabel('Team Avg Bowling Economy')
axes[1].set_title('Bowling Economy: Winners vs Losers', fontweight='bold')
axes[1].legend()

plt.suptitle('Player Stats: Do Better Stats = More Wins?', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('charts/07_player_impact.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: charts/07_player_impact.png')

## Chart 8: Season Progression
**Question:** Does batting-first advantage change as the season progresses?

In [ ]:
# Split matches into early (match 1-7), mid (8-11), and late (12+ / playoffs)
full['match_phase'] = pd.cut(
    full['match_num_in_season'],
    bins=[0, 7, 11, 200],
    labels=['Early Season\n(1-7)', 'Mid Season\n(8-11)', 'Late/Playoffs\n(12+)']
)

phase_stats = full.groupby('match_phase', observed=True).agg(
    bat_first_wr = ('team_a_won', 'mean'),
    count        = ('team_a_won', 'count')
).reset_index()

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(phase_stats['match_phase'].astype(str), phase_stats['bat_first_wr'],
              color=[ACCENT, ACCENT2, GREEN], alpha=0.85, width=0.5)
for bar, (_, row) in zip(bars, phase_stats.iterrows()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f"{row['bat_first_wr']:.1%}\n(n={int(row['count'])})",
            ha='center', fontsize=10)
ax.axhline(0.5, color='white', linestyle='--', alpha=0.4)
ax.set_ylim(0.3, 0.65)
ax.set_ylabel('Bat-First Win Rate')
ax.set_title('Bat-First Win Rate by Season Phase', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('charts/08_season_progression.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: charts/08_season_progression.png')

## Chart 9: Player Matchup Heatmap — MI vs CSK
**Question:** Which MI batters dominate CSK bowlers and vice versa?
This is exactly the **TV broadcast matchup card**.

In [ ]:
# Pick top MI batters and top CSK bowlers with meaningful history
MI_BATTERS  = ['RG Sharma', 'SR Tendulkar', 'KA Pollard', 'HH Pandya', 'Ishan Kishan',
               'AT Rayudu', 'SA Yadav', 'TM Head']
CSK_BOWLERS = ['R Ashwin', 'DJ Bravo', 'MM Sharma', 'Harbhajan Singh',
               'RA Jadeja', 'DS Kulkarni', 'Imran Tahir', 'DL Chahar']

# Filter matchup data to only MI batters vs CSK bowlers
mi_vs_csk = matchups[
    (matchups['batter'].isin(MI_BATTERS)) &
    (matchups['bowler'].isin(CSK_BOWLERS))
].copy()

# Only keep pairs with at least 6 balls faced
mi_vs_csk = mi_vs_csk[mi_vs_csk['balls_faced'] >= 6]

# Pivot into a matrix: rows = batters, columns = bowlers, values = SR
if len(mi_vs_csk) > 0:
    heatmap_data = mi_vs_csk.pivot_table(
        index='batter', columns='bowler', values='matchup_sr', aggfunc='mean'
    )

    fig, ax = plt.subplots(figsize=(12, 6))
    sns.heatmap(
        heatmap_data, annot=True, fmt='.0f', cmap='RdYlGn',
        vmin=60, vmax=200, center=130,
        ax=ax, linewidths=0.5,
        cbar_kws={'label': 'Strike Rate'}
    )
    ax.set_title('MI Batters vs CSK Bowlers — Historical Strike Rates\n(Green = batter dominates, Red = bowler dominates)',
                 fontsize=12, fontweight='bold', pad=12)
    ax.set_xlabel('CSK Bowler')
    ax.set_ylabel('MI Batter')
    plt.tight_layout()
    plt.savefig('charts/09_matchup_heatmap.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: charts/09_matchup_heatmap.png')
else:
    print('Not enough matchup data for these specific players. Try different names.')
    print('Available MI batters in matchup data:', matchups[matchups['batter'].isin(MI_BATTERS)]['batter'].unique())

## Summary of Insights

In [ ]:
print('=== KEY INSIGHTS FROM EDA ===')
print()

# 1. Toss impact
toss_wr = (full['toss_winner_is_ta'] == full['team_a_won']).mean()
print(f'1. Toss winner win rate       : {toss_wr:.1%}  → nearly useless alone')

# 2. Bat first overall
print(f'2. Bat-first win rate overall : {full["team_a_won"].mean():.1%}  → chasing slightly favoured')

# 3. Top correlated feature with winning
feature_cols_only = [c for c in full.columns if c.startswith(('ta_','tb_')) and c != 'team_a_won']
corr_with_target = full[feature_cols_only + ['team_a_won']].corr()['team_a_won'].drop('team_a_won')
top_feature = corr_with_target.abs().idxmax()
print(f'3. Most correlated feature    : {top_feature} ({corr_with_target[top_feature]:.3f})')

# 4. Charts saved
charts = os.listdir('charts')
print(f'\n{len(charts)} charts saved to notebooks/charts/')

## Your Tasks (Before Phase 6)

```
□ For each chart, write 2-3 sentences about what it tells you:
  e.g. Chart 3: 'MI has a high win rate against RCB historically.
                 CSK is the most evenly matched opponent for MI.'

□ From Chart 5 (correlation heatmap):
  - Identify 2 features that are highly correlated WITH EACH OTHER (redundant)
  - Identify the top 3 features most correlated with team_a_won

□ Identify 1 surprising insight from any chart

□ Add to interview notebook:
  Q: 'Walk me through your EDA process.'
  A: [describe what you looked at and why]

  Q: 'Does the toss matter in IPL?'
  A: [use the numbers from Chart 2 to answer this properly]
```